# Week 11

# Part A — Load a professional regression dataset
### Use UCI’s Concrete Compressive Strength dataset: 1,030 concrete mixes, eight numeric inputs, and measured strength in MPa. Inputs are seven ingredient quantities (kg/m³) and age (days). Source: Yeh (1998), UCI, doi:10.24432/C5PK67, CC BY 4.0. The first run needs internet; later runs use the saved CSV.

## Libraries

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


## UCI Concrete Compressive Strength Dataset

## sklearn imports

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

## Fetching Data

In [ ]:
WEEK_DIR = Path("week11")
for folder in ("data", "models", "reports", "figures"):
    (WEEK_DIR / folder).mkdir(parents=True, exist_ok=True)

data_path = WEEK_DIR / "data/concrete_strength.csv"
if not data_path.exists():
    url = "https://archive.ics.uci.edu/static/public/165/data.csv"
    pd.read_csv(url).to_csv(data_path, index=False)
data = pd.read_csv(data_path)
data.columns = [
    "cement_kg_m3", "slag_kg_m3", "fly_ash_kg_m3", "water_kg_m3",
    "superplasticizer_kg_m3", "coarse_aggregate_kg_m3",
    "fine_aggregate_kg_m3", "age_days", "strength_mpa"
]
print(data.head())
print(data.shape)
print(data.dtypes)
print(data.isna().sum())
print(data["strength_mpa"].describe())

   cement_kg_m3  slag_kg_m3  fly_ash_kg_m3  water_kg_m3  \
0         540.0         0.0            0.0        162.0   
1         540.0         0.0            0.0        162.0   
2         332.5       142.5            0.0        228.0   
3         332.5       142.5            0.0        228.0   
4         198.6       132.4            0.0        192.0   

   superplasticizer_kg_m3  coarse_aggregate_kg_m3  fine_aggregate_kg_m3  \
0                     2.5                  1040.0                 676.0   
1                     2.5                  1055.0                 676.0   
2                     0.0                   932.0                 594.0   
3                     0.0                   932.0                 594.0   
4                     0.0                   978.4                 825.5   

   age_days  strength_mpa  
0        28         79.99  
1        28         61.89  
2       270         40.27  
3       365         41.05  
4       360         44.30  
(1030, 9)
cement_kg_m3    

# Answer the following questions.


**A1a. Record the number of observations and input features.**

The dataset contains 1,030 observations (rows) and 8 input features (representing the 7 ingredient quantities and the age of the concrete).

**A1b. Confirm whether any values are missing.**

There are no missing values in this dataset; the output of data.isna().sum() is 0 for all columns.

**A1c. Record the target minimum, maximum, mean, and standard deviation.**

Based on the standard UCI Concrete Compressive Strength dataset, the strength_mpa column statistics are approximately:

 Minimum: 2.33 MPa

 Maximum: 82.60 MPa

 Mean: 35.82 MPa

Standard Deviation: 16.71 MPa

**A1d. Explain why the target is suitable for regression rather than classification.**

 The target variable (strength_mpa) is a continuous numerical measurement rather than a set of discrete categories. Regression algorithms are explicitly designed to predict continuous quantities on a sliding scale, whereas classification models are used to assign data to predefined, separate classes (like "pass" or "fail").

**A1e. State one limitation of predicting concrete strength from this historical dataset.**

 This dataset was published in 1998 and is restricted to the specific materials, environmental conditions, and formulations available at that time. A model trained on this data may not generalize well to modern concrete mixes that utilize newer chemical admixtures, alternative curing methods, or different aggregate qualities not represented in these historical historical records.

# Part B — One shared split and a baseline
 Define cement_kg_m3 as the single feature for Simple Linear Regression and use all eight input features for Multiple Linear Regression. Create the split once, then derive both feature sets from the same row indices.

In [ ]:
feature_columns = data.columns.drop("strength_mpa").tolist()
target_name = "strength_mpa"

train_data, test_data = train_test_split(
    data, test_size=0.20, random_state=42
)

X_train_simple = train_data[["cement_kg_m3"]]
X_test_simple = test_data[["cement_kg_m3"]]
X_train_multi = train_data[feature_columns]
X_test_multi = test_data[feature_columns]
y_train = train_data[target_name]
y_test = test_data[target_name]

assert target_name not in feature_columns
assert train_data.index.intersection(test_data.index).empty
assert list(X_train_multi.columns) == feature_columns

In [ ]:
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train_multi, y_train)
baseline_predictions = baseline.predict(X_test_multi)

# Answer the following questions.


**B1a. explain what value the baseline predicts and why a trained model must be compared with it.**

A regression baseline model (like a Dummy Regressor) simply predicts the mean (average) target value of the training data for every new observation, completely ignoring the input features. A trained model must be compared against it to establish a minimum performance threshold. If a trained machine learning model cannot produce a lower error than a naive rule that just guesses the historical average every time, it proves the model has not learned any useful predictive patterns from the data.

**B1b. If the multiple regression model performs almost the same as the mean baseline on unseen data, what does this suggest about its practical value?**

If the multiple regression model performs almost the same as the mean baseline on unseen data, it has zero practical value. This outcome suggests that the model is failing to extract any meaningful relationship between the concrete's ingredients and its final strength, making its predictions no more reliable than blindly guessing the average strength for every new mix.

# Part C — Train simple and multiple regression

## C1. Fit both models

In [ ]:
simple_model = LinearRegression()
simple_model.fit(X_train_simple, y_train)
simple_predictions = simple_model.predict(X_test_simple)

multiple_model = LinearRegression()
multiple_model.fit(X_train_multi, y_train)
multiple_predictions = multiple_model.predict(X_test_multi)

print("Simple slope:", simple_model.coef_[0])
print("Simple intercept:", simple_model.intercept_)

Simple slope: 0.07789606791104087
Simple intercept: 13.785171881915787


## C2. Interpret the simple model

# Answer the following questions.


**C2a. State the direction of the cement_kg_m3 coefficient and interpret it in MPa per kg/m³.**

The direction of the coefficient is **positive**. This means that for every additional **1 kg/m³ of cement** added to the mix, the predicted compressive strength of the concrete increases by the exact value of the coefficient in **MPa**. 

**C2b. Explain the coefficient as an association in the dataset, not a causal effect.**

The coefficient only shows a **historical pattern (correlation)**—mixes in this specific dataset that contained more cement generally happened to have higher strength. It does not prove a strict cause-and-effect rule, because other overlapping factors (like the water-to-cement ratio or curing conditions) influenced the final strength simultaneously.

**C2c. Explain why the intercept may have limited practical meaning.**

The intercept represents the mathematical prediction of concrete strength when the cement quantity is exactly **0 kg/m³**. Since actual concrete cannot exist without cement or other base ingredients, this baseline value is purely a mathematical anchor for the regression line and has **no physical, real-world meaning**.

## C3. Inspect multiple-regression coefficients

In [ ]:
coefficients = pd.DataFrame({
    "feature": feature_columns,
    "coefficient": multiple_model.coef_
}).sort_values("coefficient", key=abs, ascending=False)

print(coefficients)
coefficients.to_csv(
    WEEK_DIR / "reports/coefficients.csv", index=False
)

                  feature  coefficient
4  superplasticizer_kg_m3     0.310570
3             water_kg_m3    -0.135276
0            cement_kg_m3     0.118994
7                age_days     0.116439
1              slag_kg_m3     0.108536
2           fly_ash_kg_m3     0.082152
6    fine_aggregate_kg_m3     0.024100
5  coarse_aggregate_kg_m3     0.017867


# Part - D

## D1. Calculate consistent metrics

In [ ]:
def regression_metrics(actual, predicted):
    mse = mean_squared_error(actual, predicted)
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mse),
        "R2": r2_score(actual, predicted),
    }

results = pd.DataFrame([
    {"model": "Mean baseline", **regression_metrics(y_test, baseline_predictions)},
    {"model": "Simple linear", **regression_metrics(y_test, simple_predictions)},
    {"model": "Multiple linear", **regression_metrics(y_test, multiple_predictions)},
])
print(results.round(3))
results.to_csv(WEEK_DIR / "reports/model_comparison.csv", index=False)

             model     MAE    RMSE     R2
0    Mean baseline  13.052  16.054 -0.000
1    Simple linear  11.556  13.885  0.252
2  Multiple linear   7.746   9.796  0.628


**D1a. Which trained model beats the baseline on MAE and RMSE?**

Both the **Simple Linear Regression** and **Multiple Linear Regression** models beat the baseline. Between the two, Multiple Linear Regression achieves the lowest MAE and RMSE overall.

**D1b. How much does Multiple Linear Regression improve over Simple Linear Regression?**

Multiple Linear Regression significantly improves the predictions because concrete strength depends heavily on the interaction of all ingredients and age, not just cement. 

**D1c. If R² is negative, explain what that means without calling it negative accuracy.**

A negative R² means that the model performs **worse than simply guessing the baseline average**. It indicates that the model fails to follow the data's trend and actually introduces more error than a flat line drawn straight through the mean of the target values.

**D1d. Select a provisional model using at least two metrics.**

Select the **Multiple Linear Regression** model. It is the best choice because it achieves the **lowest MAE and RMSE** (its predictions are closest to the actual strength) and the **highest R² score** (it successfully explains the highest percentage of variance in the dataset).

## D2. Compare training And testing error

In [ ]:
train_predictions = multiple_model.predict(X_train_multi)
train_rmse = regression_metrics(y_train, train_predictions)["RMSE"]
test_rmse = regression_metrics(y_test, multiple_predictions)["RMSE"]
print(f"Training RMSE: {train_rmse:.2f}")
print(f"Testing RMSE:  {test_rmse:.2f}")

## D3. Inspect predictions and residuals

In [ ]:
evaluation = pd.DataFrame({
    "actual": y_test,
    "predicted": multiple_predictions,
})
evaluation["residual"] = evaluation["actual"] - evaluation["predicted"]
evaluation["absolute_error"] = evaluation["residual"].abs()
evaluation.to_csv(WEEK_DIR / "reports/predictions.csv", index=True)
print(evaluation.nlargest(5, "absolute_error"))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(evaluation["actual"], evaluation["predicted"], alpha=0.7)
low = min(evaluation["actual"].min(), evaluation["predicted"].min())
high = max(evaluation["actual"].max(), evaluation["predicted"].max())
axes[0].plot([low, high], [low, high], "r--")
axes[0].set(xlabel="Actual", ylabel="Predicted", title="Actual vs predicted")

axes[1].scatter(evaluation["predicted"], evaluation["residual"], alpha=0.7)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set(xlabel="Predicted", ylabel="Residual", title="Residual plot")
fig.tight_layout()
fig.savefig(WEEK_DIR / "figures/regression_diagnostics.png", dpi=300)
plt.show()